# Live experiment analysis — `spanish_basic`

Compare **baseline** (batched GPT) vs **individual** (one call per sentence) on real GPT output.

Headline metrics:
- `expected_form_match` — constraint satisfaction
- `grammar_languagetool` — grammar quality (EFSR)
- `verb_morphology` — spaCy diagnostic (not headline)

Run live experiments first:
```bash
python3 -m research.run_experiment --benchmark spanish_basic --method baseline_default --live
python3 -m research.run_experiment --benchmark spanish_basic --method individual_default --live
```

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from research.db.database import SessionLocal, init_db
from research.db.models import (
    Benchmark,
    ConstraintSet,
    Experiment,
    ExperimentMetric,
    GeneratedSentence,
    MethodConfig,
    SentenceEvaluation,
)

init_db()
session = SessionLocal()
print(f"Connected to: {session.bind.url}")

Connected to: sqlite+pysqlite:////Users/joshuagraham/Desktop/Diss/LinguistOS/research/research.db


## 1. Live experiments on `spanish_basic`

In [2]:
LIVE_SUFFIX = "_live"
BENCHMARK = "spanish_basic"

experiments = (
    session.query(Experiment)
    .join(Benchmark)
    .filter(Benchmark.name == BENCHMARK)
    .filter(Experiment.name.like(f"%{LIVE_SUFFIX}"))
    .order_by(Experiment.id)
    .all()
)

df_exp = pd.DataFrame([
    {
        "id": e.id,
        "name": e.name,
        "method": e.method_config.name if e.method_config else None,
        "generator": e.method_config.method if e.method_config else None,
        "status": e.status,
        "created_at": e.created_at,
    }
    for e in experiments
])
df_exp

,id,name,method,generator,status,created_at
0,3,baseline_gpt_spanish_basic_live,baseline_default,baseline_gpt,completed,2026-06-08 18:54:48.020241
1,4,individual_gpt_spanish_basic_live,individual_default,individual_gpt,completed,2026-06-08 18:55:04.815860


## 2. Method comparison (experiment-wide)

In [3]:
METRICS_OF_INTEREST = [
    "pass_rate::expected_form_match",
    "pass_rate::grammar_languagetool",
    "errors_per_100w::grammar_languagetool",
    "pass_rate::verb_morphology",
    "uniqueness_ratio_experiment",
]

rows = []
for e in experiments:
    for metric_name in METRICS_OF_INTEREST:
        m = (
            session.query(ExperimentMetric)
            .filter_by(
                experiment_id=e.id,
                scope="experiment",
                metric_name=metric_name,
            )
            .one_or_none()
        )
        rows.append({
            "method": e.method_config.name,
            "metric": metric_name,
            "value": m.value if m else None,
        })

df_metrics = pd.DataFrame(rows)
pivot = df_metrics.pivot(index="method", columns="metric", values="value")
pivot

metric,errors_per_100w::grammar_languagetool,pass_rate::expected_form_match,pass_rate::grammar_languagetool,pass_rate::verb_morphology,uniqueness_ratio_experiment
method,,,,,
baseline_default,0.0,1.0,1.0,0.666667,1.0000
individual_default,0.0,1.0,1.0,0.933333,0.7333


## 3. Per constraint set

In [4]:
def constraint_label(cs: ConstraintSet) -> str:
    return f"{cs.keyword} / {cs.tense} / {cs.person} / {cs.number}"

cs_rows = []
for e in experiments:
    constraint_sets = (
        session.query(ConstraintSet)
        .filter_by(benchmark_id=e.benchmark_id)
        .all()
    )
    for cs in constraint_sets:
        for metric_name in [
            "pass_rate::expected_form_match",
            "pass_rate::grammar_languagetool",
            "pass_rate::verb_morphology",
        ]:
            m = (
                session.query(ExperimentMetric)
                .filter_by(
                    experiment_id=e.id,
                    scope="constraint_set",
                    constraint_set_id=cs.id,
                    metric_name=metric_name,
                )
                .one_or_none()
            )
            cs_rows.append({
                "method": e.method_config.name,
                "constraint": constraint_label(cs),
                "metric": metric_name.replace("pass_rate::", ""),
                "pass_rate": m.value if m else None,
            })

df_cs = pd.DataFrame(cs_rows)
df_cs.pivot_table(
    index=["method", "constraint"],
    columns="metric",
    values="pass_rate",
)

metric                                                  expected_form_match  \
method             constraint                                                 
baseline_default   comer / preterite / 1st / plural                     1.0   
                   correr / present / 1st / singular                    1.0   
                   escribir / preterite / 3rd / plural                  1.0   
                   hablar / present / 2nd / singular                    1.0   
                   vivir / future / 3rd / singular                      1.0   
individual_default comer / preterite / 1st / plural                     1.0   
                   correr / present / 1st / singular                    1.0   
                   escribir / preterite / 3rd / plural                  1.0   
                   hablar / present / 2nd / singular                    1.0   
                   vivir / future / 3rd / singular                      1.0   

metric                                                  grammar_languagetool  \
method             constraint                                                  
baseline_default   comer / preterite / 1st / plural                      1.0   
                   correr / present / 1st / singular                     1.0   
                   escribir / preterite / 3rd / plural                   1.0   
                   hablar / present / 2nd / singular                     1.0   
                   vivir / future / 3rd / singular                       1.0   
individual_default comer / preterite / 1st / plural                      1.0   
                   correr / present / 1st / singular                     1.0   
                   escribir / preterite / 3rd / plural                   1.0   
                   hablar / present / 2nd / singular                     1.0   
                   vivir / future / 3rd / singular                       1.0   

metric                                                  verb_morphology  
method             constraint                                            
baseline_default   comer / preterite / 1st / plural            0.000000  
                   correr / present / 1st / singular           0.666667  
                   escribir / preterite / 3rd / plural         1.000000  
                   hablar / present / 2nd / singular           0.666667  
                   vivir / future / 3rd / singular             1.000000  
individual_default comer / preterite / 1st / plural            0.666667  
                   correr / present / 1st / singular           1.000000  
                   escribir / preterite / 3rd / plural         1.000000  
                   hablar / present / 2nd / singular           1.000000  
                   vivir / future / 3rd / singular             1.000000

## 4. Tool disagreement (sentence-level)

Buckets:
- **EF pass / LT fail** — right surface form, grammar slip (LT value-add)
- **EF pass / VM fail** — spaCy disagrees with gold (parser diagnostic)

In [5]:
def disagreement_rows(experiment: Experiment) -> list[dict]:
    out = []
    sentences = (
        session.query(GeneratedSentence)
        .filter_by(experiment_id=experiment.id)
        .order_by(GeneratedSentence.constraint_set_id, GeneratedSentence.sample_index)
        .all()
    )
    for s in sentences:
        scores = {ev.evaluator_name: ev.score for ev in s.evaluations}
        details = {ev.evaluator_name: ev.details for ev in s.evaluations}
        ef = scores.get("expected_form_match", 0.0)
        lt = scores.get("grammar_languagetool", 0.0)
        vm = scores.get("verb_morphology", 0.0)

        if ef >= 0.5 and lt < 0.5:
            bucket = "EF pass / LT fail"
        elif ef < 0.5 and lt >= 0.5:
            bucket = "EF fail / LT pass"
        elif ef >= 0.5 and vm < 0.5:
            bucket = "EF pass / VM fail"
        else:
            continue

        lt_matches = (details.get("grammar_languagetool") or {}).get("matches", [])
        out.append({
            "method": experiment.method_config.name,
            "bucket": bucket,
            "sentence": s.sentence,
            "translation": s.translation,
            "expected_form_match": ef,
            "grammar_languagetool": lt,
            "verb_morphology": vm,
            "lt_rules": [m.get("rule") for m in lt_matches],
        })
    return out

disagreements = []
for e in experiments:
    disagreements.extend(disagreement_rows(e))

df_disagree = pd.DataFrame(disagreements)
if df_disagree.empty:
    print("No EF/LT disagreements in live runs.")
else:
    display(df_disagree)

if not df_disagree.empty:
    df_disagree.groupby(["method", "bucket"]).size().unstack(fill_value=0)

,method,bucket,sentence,translation,expected_form_match,grammar_languagetool,verb_morphology,lt_rules
0,baseline_default,EF pass / VM fail,Comimos pan.,We ate bread.,1.0,1.0,0.0,[]
1,baseline_default,EF pass / VM fail,Comimos temprano.,We ate early.,1.0,1.0,0.0,[]
2,baseline_default,EF pass / VM fail,Comimos juntos ayer.,We ate together yesterday.,1.0,1.0,0.0,[]
3,baseline_default,EF pass / VM fail,¿Tú hablas conmigo?,Do you speak with me?,1.0,1.0,0.0,[]
4,baseline_default,EF pass / VM fail,Yo corro al trabajo.,I run to work.,1.0,1.0,0.0,[]
5,individual_default,EF pass / VM fail,Comimos pan.,We ate bread.,1.0,1.0,0.0,[]


## 5. All generated sentences with scores

In [6]:
sentence_rows = []
for e in experiments:
    sentences = (
        session.query(GeneratedSentence)
        .filter_by(experiment_id=e.id)
        .order_by(GeneratedSentence.constraint_set_id, GeneratedSentence.sample_index)
        .all()
    )
    for s in sentences:
        cs = s.constraint_set
        scores = {ev.evaluator_name: ev.score for ev in s.evaluations}
        sentence_rows.append({
            "experiment_id": e.id,
            "method": e.method_config.name,
            "keyword": cs.keyword,
            "expected_form": cs.expected_form,
            "sentence": s.sentence,
            "translation": s.translation,
            **{k: scores.get(k) for k in [
                "expected_form_match",
                "grammar_languagetool",
                "verb_morphology",
            ]},
        })

df_sentences = pd.DataFrame(sentence_rows)
df_sentences

,experiment_id,method,keyword,expected_form,sentence,translation,expected_form_match,grammar_languagetool,verb_morphology
0,3,baseline_default,comer,comimos,Comimos pan.,We ate bread.,1.0,1.0,0.0
1,3,baseline_default,comer,comimos,Comimos temprano.,We ate early.,1.0,1.0,0.0
2,3,baseline_default,comer,comimos,Comimos juntos ayer.,We ate together yesterday.,1.0,1.0,0.0
3,3,baseline_default,vivir,vivirá,Él vivirá aquí mañana.,He will live here tomorrow.,1.0,1.0,1.0
4,3,baseline_default,vivir,vivirá,Ella vivirá en Madrid el próximo año.,She will live in Madrid next year.,1.0,1.0,1.0
5,3,baseline_default,vivir,vivirá,Tu amigo vivirá más cerca de ti.,Your friend will live closer to you.,1.0,1.0,1.0
6,3,baseline_default,hablar,hablas,Tú hablas español.,You speak Spanish.,1.0,1.0,1.0
7,3,baseline_default,hablar,hablas,¿Tú hablas conmigo?,Do you speak with me?,1.0,1.0,0.0
8,3,baseline_default,hablar,hablas,Tú hablas claro.,You speak clearly.,1.0,1.0,1.0
9,3,baseline_default,escribir,escribieron,Ellos escribieron una carta.,They wrote a letter.,1.0,1.0,1.0


## 6. LanguageTool error breakdown

In [7]:
lt_rows = []
for e in experiments:
    m = (
        session.query(ExperimentMetric)
        .filter_by(
            experiment_id=e.id,
            scope="experiment",
            metric_name="lt_error_breakdown_experiment",
        )
        .one_or_none()
    )
    lt_rows.append({
        "method": e.method_config.name,
        "total_lt_errors": m.value if m else None,
        **(m.breakdown or {}),
    })

pd.DataFrame(lt_rows)

,method,total_lt_errors
0,baseline_default,0.0
1,individual_default,0.0


In [8]:
session.close()